# musicdetect sur Kaggle — fin de la phase 1

**Une session = un « Save & Run All (Commit) »** : il tourne en arrière-plan, fenêtre fermée, et ne risque
pas d'être tué par un redémarrage de Windows Update. 9 h au plus avec GPU, 12 h sans.

Ce notebook **ne refait aucun entraînement déjà terminé**. Il lit les modèles présents et la colonne `test`
des fichiers de métriques, puis n'exécute que ce qui manque. Interrompu, il reprend au bon endroit.

## Les deux Datasets

| Dataset | Contenu | Fréquence |
|---|---|---|
| `musicdetect-segments` | `index_fma_phase1.csv`, `index_fmc_phase1.csv`, `segments/fma_phase1/`, `segments/fmc_phase1/` (~13 Go) | **téléversé une fois** |
| `musicdetect-etat` | `models/` et `results/` (~20 Mo) | **nouvelle version à chaque session** |

Les segments ne sont pas régénérés sur Kaggle : les résultats des graines 1 et 2 ont été calculés sur
ces segments-là, et une reconstruction introduirait un risque de divergence qu'on ne saurait pas vérifier.
Les monter en lecture seule garantit que les trois graines portent sur les mêmes données.

## Avant chaque session

Panneau de droite : accélérateur **GPU T4**, **Internet On**, et les deux Datasets en entrée,
`musicdetect-etat` sur sa **dernière** version. Un secret Kaggle nommé `musicdetect` doit contenir le
JSON du jeton d'API (`{"username": "...", "key": "..."}`) pour que `save_state()` puisse publier.

> Le T4 n'apporte ici qu'un gain modeste : les goulets sont l'augmentation de vitesse de M3 et le calcul
> des empreintes à l'évaluation, tous deux sur CPU et parallélisés par `JOBS`. Si tu butes sur la limite
> de 9 h, repasse l'accélérateur à *None* : tu gagnes 3 h de session pour quelques minutes de GPU perdues.

## 0. Réglages de la session

In [ ]:
# ===== La seule cellule à modifier entre deux sessions =====
STAGE = 'all'              # 'train_eval' : entraînements + évaluations seulement
                           # 'attacks'    : grille d'attaques seulement (H4)
                           # 'report'     : rapports + comparaisons appariées seulement
                           # 'all'        : tout, dans l'ordre, sous contrainte de budget

SEEDS = [1, 2, 3]          # graines traitées ; celles qui sont finies sont sautées de toute façon
ATTACK_PRESET = 'prereg'   # 'prereg' : vitesse 0,9 / 1,1 + MP3 128  -> les 3 attaques pré-enregistrées
                           # 'full'   : les 19 attaques de default.yaml (exige librubberband pour pitch)

MAX_HOURS = 7.0            # GPU : 9 h de session, on garde une marge pour la sauvegarde
JOBS = 4                   # Kaggle donne 4 vCPU ; --jobs 1 (le défaut) est ce qui rendait M3 si lent
DEVICE = 'cuda'            # 'cuda' ou 'cpu'

REPO_URL = 'https://github.com/CedricDada/MusicDetect.git'
STATE_DATASET = 'cedricdada/musicdetect-etat'
SECRET_NAME = 'musicdetect'
CFG = 'configs/phase1.yaml'

## 1. GPU, dépôt, segments montés et état repris

In [ ]:
!nvidia-smi -L || echo "pas de GPU sur cette session"
import os, sys, glob, json, shutil, subprocess, time
from pathlib import Path

T0 = time.time()
MAX_SECONDS = int(MAX_HOURS * 3600)

BASE = Path('/kaggle/working/md')        # état : work/models et results (part dans le Dataset)
REPO = Path('/kaggle/working/repo')
WORK = BASE / 'work'
RESULTS = BASE / 'results'
for d in (WORK, RESULTS):
    d.mkdir(parents=True, exist_ok=True)

# --- dépôt : clone neuf, seule façon sûre d'avoir le dernier code
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '-q', REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
print('dépôt :', subprocess.run(['git', 'log', '--oneline', '-1'],
                                capture_output=True, text=True).stdout.strip())

# --- installation : ne poser que ce qui manque, pour ne pas remplacer le torch/numpy de Kaggle
import importlib.util
missing = [pkg for mod, pkg in (('soxr', 'soxr'), ('yaml', 'pyyaml'), ('joblib', 'joblib'),
                                ('sklearn', 'scikit-learn'), ('tqdm', 'tqdm'),
                                ('transformers', 'transformers'))
           if importlib.util.find_spec(mod) is None]
if missing:
    print('installation :', missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)

# le point d'entrée console, avec repli si pip ne l'a pas mis dans le PATH
if shutil.which('musicdetect'):
    MD = ['musicdetect']
else:
    MD = [sys.executable, '-c', 'import sys; from musicdetect.cli import main; sys.exit(main())']
print('commande :', MD[0])

In [ ]:
# --- segments : chaque élément est cherché par son nom, où que Kaggle l'ait décompressé.
# Peu importe donc que le Dataset ait les dossiers à sa racine ou sous segments/.
def find_one(pattern, what, want_dir=False):
    hits = sorted(p for p in glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
                  if (os.path.isdir(p) if want_dir else os.path.isfile(p)))
    assert hits, (f"{what} introuvable sous /kaggle/input (motif {pattern!r}). "
                  "Le Dataset musicdetect-segments est-il bien en entrée ?")
    if len(hits) > 1:
        print(f'  plusieurs {what}, on prend : {hits[0]}')
    return Path(hits[0])


# Les chemins dans l'index sont relatifs au dossier de segments, donc un lien suffit.
# Chaque graine a son propre tag, qui dérive tous les chemins : on recrée ici l'équivalent
# des jonctions Windows, pour que les trois graines partagent les mêmes WAV sans copie.
SEGS = WORK / 'segments'
SEGS.mkdir(exist_ok=True)
for kind in ('fma', 'fmc'):
    idx = find_one(f'index_{kind}_phase1.csv', f'index {kind}')
    src = find_one(f'{kind}_phase1', f'segments {kind}', want_dir=True)
    print(f'{kind} : index {idx}\n      segments {src}')
    for tag in ['', '_s1', '_s2', '_s3']:
        shutil.copy2(idx, WORK / f'index_{kind}_phase1{tag}.csv')
        dst = SEGS / f'{kind}_phase1{tag}'
        if not dst.exists():
            os.symlink(src, dst)

# Les WAV sont rangés par source (encodec_24k_6kbps/, fmc_musicgen/, ...) : comptage récursif.
n_fma = sum(1 for _ in (SEGS / 'fma_phase1').rglob('*.wav'))
n_fmc = sum(1 for _ in (SEGS / 'fmc_phase1').rglob('*.wav'))
print(f'\nsegments visibles : FMA {n_fma} (attendu 15992), FakeMusicCaps {n_fmc} (attendu 27605)')
assert (n_fma, n_fmc) == (15992, 27605), (
    "nombre de segments inattendu : Dataset incomplet ou mal décompressé. "
    "Des résultats calculés sur un sous-ensemble ne seraient pas comparables aux graines 1 et 2.")

In [ ]:
# --- état : modèles et résultats des sessions précédentes
restored = None
for cand in sorted(glob.glob('/kaggle/input/*/')):
    if Path(cand) == SEG_ROOT or (Path(cand) / 'segments').is_dir():
        continue
    src_models, src_results = Path(cand) / 'models', Path(cand) / 'results'
    if src_models.is_dir() or src_results.is_dir():
        if src_models.is_dir():
            shutil.copytree(src_models, WORK / 'models', dirs_exist_ok=True)
        if src_results.is_dir():
            shutil.copytree(src_results, RESULTS, dirs_exist_ok=True)
        restored = cand
        break
print('état repris depuis :', restored or "RIEN : seules les étapes neuves tourneront")
for tag in ('phase1', 'phase1_s1', 'phase1_s2', 'phase1_s3'):
    mods = sorted(p.name for p in (WORK / 'models' / tag).glob('*')) if (WORK / 'models' / tag).is_dir() else []
    print(f'  {tag:12s} modèles : {len(mods)}')


def save_state(note):
    # Publie models/ et results/ comme nouvelle version du Dataset. N'interrompt jamais le notebook.
    try:
        from kaggle_secrets import UserSecretsClient
        tok = json.loads(UserSecretsClient().get_secret(SECRET_NAME))
        os.environ['KAGGLE_USERNAME'], os.environ['KAGGLE_KEY'] = tok['username'], tok['key']
        stage = Path('/kaggle/working/_state')
        shutil.rmtree(stage, ignore_errors=True)
        stage.mkdir(parents=True)
        if (WORK / 'models').is_dir():
            shutil.copytree(WORK / 'models', stage / 'models', dirs_exist_ok=True)
        if RESULTS.is_dir():
            shutil.copytree(RESULTS, stage / 'results', dirs_exist_ok=True)
        (stage / 'dataset-metadata.json').write_text(json.dumps(
            {"title": "musicdetect etat", "id": STATE_DATASET,
             "licenses": [{"name": "CC0-1.0"}]}), encoding='utf-8')
        r = subprocess.run(['kaggle', 'datasets', 'version', '-p', str(stage), '-m', note, '-d'],
                           capture_output=True, text=True)
        print(r.stdout or r.stderr)
    except Exception as e:
        print('sauvegarde impossible :', e)
        print('replis : les fichiers restent dans /kaggle/working/md, récupérables dans l\'Output.')

## 2. Capacités de ffmpeg et grille d'attaques

`speed` passe par `soxr` (paquet pip), mais `pitch` exige le filtre `rubberband` de ffmpeg et `mp3` exige
`libmp3lame`. Si `librubberband` manque, le preset `full` est automatiquement ramené à ce qui est faisable
plutôt que d'échouer au milieu de la grille.

In [ ]:
def ffmpeg_has(name):
    try:
        out = subprocess.run(['ffmpeg', '-hide_banner', '-filters'], capture_output=True, text=True).stdout
        enc = subprocess.run(['ffmpeg', '-hide_banner', '-encoders'], capture_output=True, text=True).stdout
        return name in out or name in enc
    except FileNotFoundError:
        return False


HAS_RB = ffmpeg_has('rubberband')
HAS_MP3 = ffmpeg_has('libmp3lame')
print(f'ffmpeg : rubberband {"oui" if HAS_RB else "NON"}, libmp3lame {"oui" if HAS_MP3 else "NON"}')

if ATTACK_PRESET == 'prereg':
    grid = {'speed': [0.9, 1.1], 'speed_extended': [], 'pitch_semitones': [],
            'mp3_kbps': [128] if HAS_MP3 else [], 'resample_hz': [], 'noise_snr_db': []}
else:
    grid = {'speed': [0.9, 1.1], 'speed_extended': [0.7, 0.8, 1.25, 1.4],
            'pitch_semitones': [-2, -1, 1, 2] if HAS_RB else [],
            'mp3_kbps': [64, 128, 192, 320] if HAS_MP3 else [],
            'resample_hz': [12000, 11025], 'noise_snr_db': [30, 20, 10]}

ATTACK_SET = [f'--set=eval.attacks.{k}={json.dumps(v)}' for k, v in grid.items()]
n_att = sum(len(v) for v in grid.values())
print(f'grille retenue : {n_att} attaques')
for k, v in grid.items():
    if v:
        print(f'  {k}: {v}')
if not HAS_RB and ATTACK_PRESET == 'full':
    print("\nAVERTISSEMENT : pitch retiré faute de librubberband. À signaler dans le rapport\n"
          "si tu comptais l'utiliser, ou installer un ffmpeg statique avant de relancer.")

## 3. Ce qu'il reste à faire

In [ ]:
import pandas as pd

GENS = ['fmc:musicgen', 'fmc:stable_audio_open', 'fmc:audioldm2', 'fmc:musicldm', 'fmc:mustango']
CODECS = ['encodec_24k_6kbps', 'encodec_24k_12kbps']
COMMON = ['-c', CFG,
          f'--set=paths.work_dir={WORK}',
          f'--set=paths.results_dir={RESULTS}']


def tests_for(codec):
    # Le controle en-domaine du codec lui-meme manquait aux evaluations 12 kbps faites en local.
    t = ['fma:encodec_24k_6kbps', 'fma:identity_24k']
    if codec != 'encodec_24k_6kbps':
        t.append(f'fma:{codec}')
    return t + GENS


def test_args(tests):
    out = []
    for t in tests:
        out += ['--test', t]
    return out


def eval_done(tag, stem, tests):
    p = RESULTS / tag / f'{stem}.csv'
    if not p.exists():
        return False
    try:
        have = set(pd.read_csv(p)['test'])
    except Exception:
        return False
    return all(t in have for t in tests)


def model_done(tag, detector, codec):
    return (WORK / 'models' / tag / f'{detector}__{codec}.pt').exists()


def plan():
    todo = []
    for s in SEEDS:
        tag = f'phase1_s{s}'
        for c in CODECS:
            for d in ('m2', 'm3'):
                if not model_done(tag, d, c):
                    todo.append(('train', tag, d, c))
                if not eval_done(tag, f'{d}__{c}', tests_for(c)):
                    todo.append(('evaluate', tag, d, c))
    for c in CODECS:
        for stem, det in (('m0-residual', 'm0'), ('m1', 'm1')):
            if not eval_done('phase1', f'{stem}__{c}', tests_for(c)):
                todo.append(('evaluate', 'phase1', det, c))
    if not (RESULTS / 'phase1' / 'attacks_m1__encodec_24k_6kbps.csv').exists():
        todo.append(('attacks', 'phase1', 'm1', 'encodec_24k_6kbps'))
    for d in ('m3', 'm2'):
        if not (RESULTS / 'phase1_s1' / f'attacks_{d}__encodec_24k_6kbps.csv').exists():
            todo.append(('attacks', 'phase1_s1', d, 'encodec_24k_6kbps'))
    return todo


print('RESTE À FAIRE')
for step in plan():
    print('  ', '  '.join(step))
print(f'\ntotal : {len(plan())} étapes (hors rapports et comparaisons)')

In [ ]:
def budget_left():
    return MAX_SECONDS - (time.time() - T0)


def run(label, args):
    # Execute une commande musicdetect en respectant le budget. Renvoie True si elle a reussi.
    if budget_left() <= 0:
        print(f'[BUDGET] {label} reporté à la prochaine session')
        return False
    print(f'\n=== {label}   (budget restant {budget_left() / 3600:.1f} h)', flush=True)
    t = time.time()
    r = subprocess.run(MD + args, stdout=sys.stdout, stderr=subprocess.STDOUT)
    ok = r.returncode == 0
    print(f"--- {'OK' if ok else 'ECHEC ' + str(r.returncode)}  {label}  "
          f"({(time.time() - t) / 60:.1f} min)", flush=True)
    if not ok:
        FAILURES.append(label)
    return ok


FAILURES = []

## 4. Entraînements et évaluations

`--jobs` parallélise l'augmentation de vitesse de M3 et le calcul des empreintes. Sa valeur par défaut est
**1**, ce qui explique les 1 h 39 observées en local sur la première augmentation.

In [ ]:
if STAGE in ('train_eval', 'all'):
    for s in SEEDS:
        tag = f'phase1_s{s}'
        for c in CODECS:
            for d in ('m2', 'm3'):
                if model_done(tag, d, c):
                    print(f'[SAUTE] train    {d} {c} {tag}')
                else:
                    run(f'train    {d} / {c} / {tag}',
                        ['train', '--detector', d, '--codec', c, '--set', f'{d}.seed={s}',
                         '--tag', tag, '--device', DEVICE, '--jobs', str(JOBS)] + COMMON)

                tests = tests_for(c)
                if eval_done(tag, f'{d}__{c}', tests):
                    print(f'[SAUTE] evaluate {d} {c} {tag}')
                elif model_done(tag, d, c):
                    run(f'evaluate {d} / {c} / {tag}',
                        ['evaluate', '--detector', d, '--codec', c] + test_args(tests)
                        + ['--tag', tag, '--device', DEVICE, '--jobs', str(JOBS)] + COMMON)
                else:
                    print(f'[SAUTE] evaluate {d} {c} {tag} : modèle absent')

    # M0 et M1 : le contrôle en-domaine 12 kbps n'avait jamais été mesuré
    for c in CODECS:
        tests = tests_for(c)
        for stem, det, extra in (('m0-residual', 'm0', ['--variant', 'residual']), ('m1', 'm1', [])):
            if eval_done('phase1', f'{stem}__{c}', tests):
                print(f'[SAUTE] evaluate {stem} {c} phase1')
            else:
                run(f'evaluate {stem} / {c} / phase1',
                    ['evaluate', '--detector', det] + extra + ['--codec', c] + test_args(tests)
                    + ['--tag', 'phase1', '--jobs', str(JOBS)] + COMMON)
else:
    print(f"étape sautée (STAGE = {STAGE!r})")

## 5. Attaques — H4

On passe la **liste complète** des tests et on désigne la cible des attaques avec `--attack-on`. Sans ça,
`evaluate` réécrirait `results/<tag>/<stem>.csv` avec la seule ligne du test demandé et le rapport perdrait
les quatre autres générateurs ainsi que le contrôle `identity`.

In [ ]:
if STAGE in ('attacks', 'all'):
    six = test_args(tests_for('encodec_24k_6kbps'))
    jobs = ['--jobs', str(JOBS)]
    for tag, d in (('phase1', 'm1'), ('phase1_s1', 'm3'), ('phase1_s1', 'm2')):
        out = RESULTS / tag / f'attacks_{d}__encodec_24k_6kbps.csv'
        if out.exists():
            print(f'[SAUTE] attaques {d} {tag}')
            continue
        if not (d in ('m0', 'm1') or model_done(tag, d, 'encodec_24k_6kbps')):
            print(f'[SAUTE] attaques {d} {tag} : modèle absent')
            continue
        run(f'attaques {d} / {tag}',
            ['evaluate', '--detector', d, '--codec', 'encodec_24k_6kbps'] + six
            + ['--attacks', '--attack-on', 'fmc:musicgen', '--tag', tag,
               '--device', DEVICE] + jobs + ATTACK_SET + COMMON)
else:
    print(f"étape sautée (STAGE = {STAGE!r})")

## 6. Rapports et comparaisons appariées M1 contre M0

In [ ]:
if STAGE in ('report', 'all'):
    for tag in ('phase1', 'phase1_s1', 'phase1_s2', 'phase1_s3'):
        if (RESULTS / tag).is_dir():
            run(f'report {tag}', ['report', '--tag', tag] + COMMON)

    sc = RESULTS / 'phase1' / 'scores'
    for g in ('musicgen', 'stable_audio_open', 'audioldm2', 'musicldm', 'mustango'):
        a = sc / f'm1__encodec_24k_6kbps__fmc_{g}.csv'
        b = sc / f'm0-residual__encodec_24k_6kbps__fmc_{g}.csv'
        if a.exists() and b.exists():
            print(f'\n===== {g} =====')
            run(f'compare {g}', ['compare', str(a), str(b), '--bootstrap', '1000'] + COMMON)
        else:
            print(f'[SAUTE] compare {g} : scores absents')
else:
    print(f"étape sautée (STAGE = {STAGE!r})")

### Agrégation multi-graines

Moyenne ± écart-type sur les graines, **séparément** des intervalles bootstrap : ce sont deux
quantités différentes, et pour M2 l'écart entre graines dépasse la largeur de l'intervalle bootstrap.

In [ ]:
frames = []
for s in (1, 2, 3):
    p = RESULTS / f'phase1_s{s}' / 'summary.csv'
    if p.exists():
        df = pd.read_csv(p)
        df['seed'] = s
        frames.append(df)

if frames:
    allsum = pd.concat(frames, ignore_index=True)
    key = ['detector', 'train_codec', 'test']
    agg = (allsum.groupby(key)['auc']
           .agg(n_graines='count', auc_moy='mean', auc_ecart='std')
           .reset_index().sort_values(key))
    agg['auc_moy'] = agg['auc_moy'].round(3)
    agg['auc_ecart'] = agg['auc_ecart'].round(3)
    out = RESULTS / 'agregat_graines.csv'
    agg.to_csv(out, index=False)
    print(f'écrit : {out}\n')
    with pd.option_context('display.width', 140, 'display.max_rows', 200):
        print(agg.to_string(index=False))
else:
    print("aucun summary.csv par graine : lance l'étape 6 d'abord")

## 7. Sauvegarde de la session

In [ ]:
save_state(f'STAGE={STAGE} seeds={SEEDS} preset={ATTACK_PRESET}')

print('\n' + '#' * 60)
print(f'durée : {(time.time() - T0) / 3600:.2f} h sur {MAX_HOURS} h de budget')
print(f'échecs : {len(FAILURES)}')
for f in FAILURES:
    print('  -', f)
reste = plan()
print(f'\nreste après cette session : {len(reste)} étapes')
for step in reste:
    print('  ', '  '.join(step))
print('#' * 60)